In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/credit-risk-customers/credit_customers.csv


## Necessary Imports

In [2]:
%pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - \ done
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=a8957c4cef1d46266139a152a41c0ef157d41878030a075483eb80d08e9ec412
  Stored in directory: /root/.cache/pip/wheels/80/1d/60/2c256ed38dddce2fdd93be545214a63e02fbd8d74fb0b7f3a6
Successfully built pyspark
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.feature import StringIndexer, VectorIndexer, VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pyspark.sql.functions as f
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession \
    .builder \
    .appName("yet_another_credit_risk_analysis") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/06/14 10:27:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.sparkContext.setLogLevel("ERROR")

## All about the data

In [6]:
df = spark.read.csv("/kaggle/input/credit-risk-customers/credit_customers.csv", inferSchema=True, header=True)
df.toPandas()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6.0,critical/other existing credit,radio/tv,1169.0,no known savings,>=7,4.0,male single,none,...,real estate,67.0,none,own,2.0,skilled,1.0,yes,yes,good
1,0<=X<200,48.0,existing paid,radio/tv,5951.0,<100,1<=X<4,2.0,female div/dep/mar,none,...,real estate,22.0,none,own,1.0,skilled,1.0,none,yes,bad
2,no checking,12.0,critical/other existing credit,education,2096.0,<100,4<=X<7,2.0,male single,none,...,real estate,49.0,none,own,1.0,unskilled resident,2.0,none,yes,good
3,<0,42.0,existing paid,furniture/equipment,7882.0,<100,4<=X<7,2.0,male single,guarantor,...,life insurance,45.0,none,for free,1.0,skilled,2.0,none,yes,good
4,<0,24.0,delayed previously,new car,4870.0,<100,1<=X<4,3.0,male single,none,...,no known property,53.0,none,for free,2.0,skilled,2.0,none,yes,bad
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,no checking,12.0,existing paid,furniture/equipment,1736.0,<100,4<=X<7,3.0,female div/dep/mar,none,...,real estate,31.0,none,own,1.0,unskilled resident,1.0,none,yes,good
996,<0,30.0,existing paid,used car,3857.0,<100,1<=X<4,4.0,male div/sep,none,...,life insurance,40.0,none,own,1.0,high qualif/self emp/mgmt,1.0,yes,yes,good
997,no checking,12.0,existing paid,radio/tv,804.0,<100,>=7,4.0,male single,none,...,car,38.0,none,own,1.0,skilled,1.0,none,yes,good
998,<0,45.0,existing paid,radio/tv,1845.0,<100,1<=X<4,4.0,male single,none,...,no known property,23.0,none,for free,1.0,skilled,1.0,yes,yes,bad


In [7]:
df.printSchema()

root
 |-- checking_status: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- credit_history: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- credit_amount: double (nullable = true)
 |-- savings_status: string (nullable = true)
 |-- employment: string (nullable = true)
 |-- installment_commitment: double (nullable = true)
 |-- personal_status: string (nullable = true)
 |-- other_parties: string (nullable = true)
 |-- residence_since: double (nullable = true)
 |-- property_magnitude: string (nullable = true)
 |-- age: double (nullable = true)
 |-- other_payment_plans: string (nullable = true)
 |-- housing: string (nullable = true)
 |-- existing_credits: double (nullable = true)
 |-- job: string (nullable = true)
 |-- num_dependents: double (nullable = true)
 |-- own_telephone: string (nullable = true)
 |-- foreign_worker: string (nullable = true)
 |-- class: string (nullable = true)



In [8]:
#list of columns
columns_list_all = list(df.toPandas().columns)

Here, we look at what kind of values each column has, and what is the datatype. \
Reference: https://stackoverflow.com/questions/78371959/pyspark-drop-duplicates-when-a-column-is-null

In [9]:
for column in columns_list_all:
    df.select(f.col(column)).printSchema()
    df.select(column).dropDuplicates([column]).show()
    df.select(f.col(column)).describe().show()

root
 |-- checking_status: string (nullable = true)

+---------------+
|checking_status|
+---------------+
|       0<=X<200|
|          >=200|
|             <0|
|    no checking|
+---------------+

+-------+---------------+
|summary|checking_status|
+-------+---------------+
|  count|           1000|
|   mean|           NULL|
| stddev|           NULL|
|    min|       0<=X<200|
|    max|    no checking|
+-------+---------------+

root
 |-- duration: double (nullable = true)

+--------+
|duration|
+--------+
|     8.0|
|     7.0|
|    47.0|
|    42.0|
|    18.0|
|    39.0|
|    36.0|
|     4.0|
|    45.0|
|    11.0|
|    21.0|
|    72.0|
|    14.0|
|    48.0|
|    22.0|
|    60.0|
|    28.0|
|    27.0|
|    10.0|
|    30.0|
+--------+
only showing top 20 rows

+-------+------------------+
|summary|          duration|
+-------+------------------+
|  count|              1000|
|   mean|            20.903|
| stddev|12.058814452756371|
|    min|               4.0|
|    max|              72.0|

We check whether there are any missing values.

In [10]:
# any missing values?
df.select([f.count(f.when(f.isnan(c), c)).alias(c) for c in columns_list_all]).toPandas()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Seems like there are no missing values.

Next, we transform the data, as it is currently not suitable to be fit to the model.

In [11]:
print(len(columns_list_all[:-1]))

20


In [12]:
#first, we index each string-type column in the dataset
stringIndexerList = [StringIndexer(inputCol=col, outputCol=col+"_index").fit(df) for col in columns_list_all]
pipeline_intermediate = Pipeline(stages=stringIndexerList)
df_r = pipeline_intermediate.fit(df).transform(df)

#next, we coagulate all the features into single vectors
featureVectorAssembler = VectorAssembler(inputCols=[col+"_index" for col in columns_list_all[:-1]], outputCol="features").transform(df_r)

# Automatically identify categorical features, and index them.
# Set maxCategories so features with > 10 distinct values are treated as continuous.
featureIndexer =\
    VectorIndexer(inputCol="features", outputCol="indexedFeatures", maxCategories=10).fit(featureVectorAssembler.select("features", "class_index"))\

final_df = featureIndexer.transform(featureVectorAssembler.select("features", "class_index")).select("indexedFeatures", "class_index")

# final_df.toPandas()

# splitting the data into training and test sets (set aside 30% for testing)
(training_df, testing_df) = final_df.randomSplit([0.75, 0.25])

## Model Training

In [13]:
training_df.toPandas()

,indexedFeatures,class_index
0,"(2.0, 19.0, 1.0, 3.0, 670.0, 2.0, 4.0, 2.0, 0....",0.0
1,"(2.0, 7.0, 4.0, 1.0, 864.0, 3.0, 3.0, 3.0, 1.0...",0.0
2,"(1.0, 8.0, 4.0, 4.0, 858.0, 1.0, 3.0, 1.0, 0.0...",0.0
3,"(2.0, 9.0, 1.0, 2.0, 470.0, 4.0, 2.0, 2.0, 0.0...",0.0
4,"(2.0, 1.0, 1.0, 3.0, 291.0, 2.0, 3.0, 2.0, 0.0...",0.0
...,...,...
760,"[3.0, 1.0, 1.0, 1.0, 903.0, 3.0, 2.0, 0.0, 2.0...",1.0
761,"[3.0, 4.0, 1.0, 1.0, 169.0, 0.0, 0.0, 3.0, 0.0...",0.0
762,"[3.0, 4.0, 1.0, 1.0, 179.0, 2.0, 1.0, 1.0, 3.0...",0.0
763,"[3.0, 10.0, 0.0, 1.0, 14.0, 2.0, 1.0, 3.0, 1.0...",1.0


In [14]:
layers = [20, 40, 20, 2]

# create the trainer and set its parameters
trainer = MultilayerPerceptronClassifier(layers=layers, blockSize=64, featuresCol='indexedFeatures', labelCol='class_index', solver='gd')

model = trainer.fit(training_df)

predictions = model.transform(testing_df)

## Model Evaluation

In [15]:
predictions_and_labels = predictions.select("indexedFeatures", "prediction", "class_index")
predictions_and_labels.toPandas()

,indexedFeatures,prediction,class_index
0,"(2.0, 2.0, 4.0, 1.0, 390.0, 2.0, 3.0, 2.0, 1.0...",0.0,1.0
1,"(1.0, 8.0, 1.0, 3.0, 746.0, 2.0, 2.0, 3.0, 2.0...",0.0,0.0
2,"(2.0, 6.0, 3.0, 1.0, 213.0, 2.0, 2.0, 1.0, 0.0...",0.0,1.0
3,"(2.0, 1.0, 1.0, 1.0, 340.0, 2.0, 3.0, 0.0, 0.0...",0.0,0.0
4,"(2.0, 2.0, 2.0, 4.0, 423.0, 1.0, 1.0, 0.0, 0.0...",0.0,0.0
...,...,...,...
230,"[2.0, 1.0, 1.0, 1.0, 530.0, 0.0, 3.0, 3.0, 0.0...",0.0,0.0
231,"[2.0, 2.0, 1.0, 4.0, 55.0, 0.0, 4.0, 2.0, 2.0,...",0.0,0.0
232,"[2.0, 4.0, 2.0, 2.0, 89.0, 0.0, 4.0, 0.0, 0.0,...",0.0,0.0
233,"[2.0, 6.0, 1.0, 5.0, 111.0, 4.0, 1.0, 0.0, 0.0...",0.0,1.0


In [16]:
# Select (prediction, true label) and compute test error
evaluator = MulticlassClassificationEvaluator(
    labelCol="class_index", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions_and_labels)
print("Test Error = %g" % (1.0 - accuracy))

Test Error = 0.276596


In [17]:
gbtModel = model.summary
print(gbtModel)  # summary only

<bound method MultilayerPerceptronClassificationModel.summary of MultilayerPerceptronClassificationModel: uid=MultilayerPerceptronClassifier_8fce25c76741, numLayers=4, numClasses=2, numFeatures=20>


## Ending the session

In [18]:
# spark.stop()